# 02 – Del LLM al clasificador

**Tarea 1 – Noticias peruanas: del titular al clasificador** · Diplomado QLAB · Opción A (La República)

Flujo del notebook:

0. **Partición 70/15/15** estratificada con `random_state=42`. Se hace **primero** para que el conjunto de test quede aislado desde el inicio y nunca se use para entrenar ni para elegir hiperparámetros.
1. **Línea base zero-shot** (sin entrenamiento) sobre 40 titulares del test.
2. **Supervisión débil con un LLM**: el LLM etiqueta todo el dataset → `datos/etiquetas_llm.csv`, más una **auditoría humana** de 20 casos.
3. **Fine-tuning** de `distilbert-base-multilingual-cased` entrenado con las **etiquetas del LLM** (supervisión débil).
4. **Evaluación final en test** contra la etiqueta de referencia (`etiqueta`, la sección editorial del diario).
5. **Tabla comparativa** y decisión.

> **Etiqueta de referencia ("gold")**: la columna `etiqueta` de `noticias.csv`, que viene de la sección donde el propio diario publicó la noticia. Es la verdad contra la que se evalúan los 3 modelos.

In [ ]:
import json, os, random, time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
DATOS = Path("datos"); MODELO_DIR = Path("modelo")
MODELO_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATOS / "noticias.csv")
print("Noticias:", df.shape)
df["etiqueta"].value_counts()

## 0. Partición estratificada 70 / 15 / 15

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["etiqueta"], random_state=SEED)

particion = pd.concat([train_df.assign(particion="train"), val_df.assign(particion="val"),
                       test_df.assign(particion="test")])
particion[["url", "particion"]].to_csv(DATOS / "particion.csv", index=False)

resumen = pd.crosstab(particion["etiqueta"], particion["particion"])[["train", "val", "test"]]
resumen.loc["TOTAL"] = resumen.sum()
for p in ["train", "val", "test"]:
    print(f"{p:<5}: {len(particion[particion.particion == p]):>4} filas ({len(particion[particion.particion == p]) / len(df):.0%})")
# Ninguna URL puede estar en dos particiones a la vez
assert not set(train_df.url) & set(test_df.url) and not set(val_df.url) & set(test_df.url)
resumen

## 1. Línea base zero-shot (sin fine-tuning)

Un modelo NLI (inferencia de lenguaje natural) decide qué etiqueta "encaja" mejor con la frase *"Esta noticia trata sobre {categoría}"*. No se entrena: es el **piso mínimo** a superar. Se evalúa sobre **40 titulares del test** (10 por categoría).

`Recognai/bert-base-spanish-wwm-cased-xnli` es más liviano y rápido en CPU; si quieres la otra opción del enunciado cambia `MODELO_ZS` a `"joeddav/xlm-roberta-large-xnli"` (≈2 GB).

In [ ]:
from transformers import pipeline

MODELO_ZS = "Recognai/bert-base-spanish-wwm-cased-xnli"
ETIQUETAS_ZS = {"política": "politica", "economía": "economia", "deportes": "deportes",
                "ciencia y tecnología": "ciencia_tecnologia"}

zs_muestra = (test_df.groupby("etiqueta", group_keys=False)
                     .apply(lambda g: g.sample(min(len(g), 10), random_state=SEED))
                     .reset_index(drop=True))

zero_shot = pipeline("zero-shot-classification", model=MODELO_ZS, device=-1)
t0 = time.perf_counter()
salidas = zero_shot(zs_muestra["titular"].tolist(), candidate_labels=list(ETIQUETAS_ZS),
                    hypothesis_template="Esta noticia trata sobre {}.")
lat_zs = (time.perf_counter() - t0) / len(zs_muestra) * 1000
zs_muestra["pred_zero_shot"] = [ETIQUETAS_ZS[s["labels"][0]] for s in salidas]

acc_zs = accuracy_score(zs_muestra["etiqueta"], zs_muestra["pred_zero_shot"])
f1_zs = f1_score(zs_muestra["etiqueta"], zs_muestra["pred_zero_shot"], average="macro")
print(f"ZERO-SHOT ({len(zs_muestra)} titulares) -> Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f} | {lat_zs:.0f} ms/titular")
print(classification_report(zs_muestra["etiqueta"], zs_muestra["pred_zero_shot"], zero_division=0))

## 2. Supervisión débil con un LLM

El LLM recibe titular + bajada y devuelve, en JSON, una categoría y su confianza. Se envían lotes de 25 noticias para gastar pocas llamadas.

**Configuración (elige una):**
- **Gemini API (recomendado, gratis):** crea una clave en https://aistudio.google.com/apikey y guárdala en un archivo `.env` dentro de `tarea_1/` con la línea `GEMINI_API_KEY=tu_clave`. El `.env` está en `.gitignore`: **la clave nunca se sube a GitHub**.
- **Ollama (local, sin internet):** instala Ollama, ejecuta `ollama pull qwen2.5:7b` y cambia `PROVEEDOR_LLM = "ollama"`.

Si `datos/etiquetas_llm.csv` ya existe, la celda lo reutiliza y no vuelve a llamar al LLM.

In [ ]:
import re
import requests
from dotenv import load_dotenv
load_dotenv()

PROVEEDOR_LLM = "gemini"            # "gemini" u "ollama"
MODELO_GEMINI = "gemini-2.5-flash"  # si da error de modelo, revisa el nombre vigente en https://ai.google.dev/gemini-api/docs/models
MODELO_OLLAMA = "qwen2.5:7b"
TAM_LOTE = 25
FORZAR_LLM = False                  # True = volver a etiquetar aunque exista el CSV
RUTA_LLM = DATOS / "etiquetas_llm.csv"

PROMPT = '''Eres editor de un diario peruano. Clasifica cada noticia en UNA sola de estas 4 categorías:
- politica: Gobierno, Presidencia, Congreso/Senado, partidos, elecciones, JNE/ONPE, Fiscalía, Poder Judicial, corrupción, Fuerzas Armadas y Policía como instituciones del Estado.
- economia: mercados, empresas, empleo, sueldos, precios, inflación, impuestos, AFP, bancos, comercio, presupuesto público, minería y agro como negocio, finanzas personales.
- deportes: fútbol y cualquier otro deporte, clubes, selecciones, atletas, torneos y resultados.
- ciencia_tecnologia: investigación científica, astronomía, biología, arqueología, medio ambiente y naturaleza, salud desde la ciencia, tecnología, internet, apps, inteligencia artificial, videojuegos.

Reglas:
1. Usa exactamente uno de estos valores: politica, economia, deportes, ciencia_tecnologia.
2. "confianza" es un número entre 0 y 1 que indica qué tan seguro estás.
3. Responde SOLO con JSON con esta forma: {"resultados": [{"id": 0, "etiqueta": "politica", "confianza": 0.9}]}

Noticias:
'''

def llamar_llm(texto):
    if PROVEEDOR_LLM == "gemini":
        from google import genai
        from google.genai import types
        clave = os.environ.get("GEMINI_API_KEY")
        if not clave:
            raise RuntimeError("Falta GEMINI_API_KEY. Créala en https://aistudio.google.com/apikey y ponla en tarea_1/.env")
        cliente = genai.Client(api_key=clave)
        resp = cliente.models.generate_content(
            model=MODELO_GEMINI, contents=texto,
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
        return resp.text
    resp = requests.post("http://localhost:11434/api/chat", timeout=900, json={
        "model": MODELO_OLLAMA, "messages": [{"role": "user", "content": texto}],
        "format": "json", "stream": False, "options": {"temperature": 0}})
    resp.raise_for_status()
    return resp.json()["message"]["content"]

def etiquetar_lote(lote):
    noticias = [{"id": int(i), "titular": r.titular, "bajada": r.bajada} for i, r in lote.iterrows()]
    texto = PROMPT + json.dumps(noticias, ensure_ascii=False, indent=0)
    for intento in range(2):  # máximo 1 reintento
        try:
            crudo = llamar_llm(texto)
            crudo = re.sub(r"^```(json)?|```$", "", crudo.strip()).strip()
            datos = json.loads(crudo)
            lista = datos["resultados"] if isinstance(datos, dict) else datos
            return {int(d["id"]): (str(d["etiqueta"]).strip().lower(), float(d.get("confianza", 0))) for d in lista}
        except Exception as e:
            print(f"  lote falló ({type(e).__name__}: {e}); reintento {intento + 1}")
            time.sleep(10)
    return {}

if RUTA_LLM.exists() and not FORZAR_LLM:
    llm_df = pd.read_csv(RUTA_LLM)
    print(f"Usando etiquetas existentes de {RUTA_LLM} ({len(llm_df)} filas)")
    lat_llm = np.nan
else:
    modelo_llm = MODELO_GEMINI if PROVEEDOR_LLM == "gemini" else MODELO_OLLAMA
    resultados = {}
    t0 = time.perf_counter()
    for inicio in range(0, len(df), TAM_LOTE):
        lote = df.iloc[inicio:inicio + TAM_LOTE]
        resultados.update(etiquetar_lote(lote))
        print(f"  {min(inicio + TAM_LOTE, len(df))}/{len(df)} noticias etiquetadas")
        time.sleep(4 if PROVEEDOR_LLM == "gemini" else 0)  # respeta el límite de peticiones por minuto
    lat_llm = (time.perf_counter() - t0) / len(df) * 1000
    hoy = datetime.now().strftime("%Y-%m-%d")
    llm_df = pd.DataFrame({
        "titular": df["titular"],
        "etiqueta_llm": [resultados.get(i, ("sin_respuesta", 0.0))[0] for i in df.index],
        "confianza": [round(resultados.get(i, ("", 0.0))[1], 3) for i in df.index],
        "modelo_llm": modelo_llm,
        "fecha": hoy,
    })
    llm_df.to_csv(RUTA_LLM, index=False, encoding="utf-8")
    (MODELO_DIR / "latencia_llm.json").write_text(json.dumps({"ms_por_titular": lat_llm}))
    print(f"Guardado {RUTA_LLM} · {lat_llm:.0f} ms por noticia (incluye pausas)")

ruta_lat = MODELO_DIR / "latencia_llm.json"
if ruta_lat.exists():
    lat_llm = json.loads(ruta_lat.read_text())["ms_por_titular"]

invalidas = ~llm_df["etiqueta_llm"].isin(CATEGORIAS)
print(f"Etiquetas fuera de las 4 categorías o sin respuesta: {invalidas.sum()}")
print(llm_df["etiqueta_llm"].value_counts())
llm_df.head()

In [ ]:
# Unimos las etiquetas del LLM al dataset (por titular, que es único tras la limpieza)
quitar = lambda d: d.drop(columns=[c for c in ["etiqueta_llm", "confianza"] if c in d.columns])
df, train_df, val_df, test_df = map(quitar, [df, train_df, val_df, test_df])
df = df.merge(llm_df[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left")
train_df = train_df.merge(llm_df[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left")
val_df = val_df.merge(llm_df[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left")
test_df = test_df.merge(llm_df[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left")

concordancia = (df["etiqueta_llm"] == df["etiqueta"]).mean()
print(f"Concordancia LLM vs. sección del diario (todo el dataset): {concordancia:.1%}")
print("Confianza media cuando acierta  :", round(df.loc[df.etiqueta_llm == df.etiqueta, "confianza"].mean(), 3))
print("Confianza media cuando se aleja :", round(df.loc[df.etiqueta_llm != df.etiqueta, "confianza"].mean(), 3))
pd.crosstab(df["etiqueta"], df["etiqueta_llm"], rownames=["sección del diario"], colnames=["LLM"])

### Auditoría humana de 20 predicciones al azar

1. La primera vez, esta celda crea `datos/auditoria_llm.csv` con 20 noticias al azar (`random_state=42`).
2. **Ábrelo en VS Code o Excel**, lee titular y bajada y escribe en `revision_humana` **si** (estás de acuerdo con `etiqueta_llm`) o **no**. Si es "no", escribe en `comentario` la categoría que tú pondrías.
3. Guarda el archivo y **vuelve a ejecutar esta celda**: calcula el porcentaje de acuerdo.

> Importante: esta revisión la tienes que hacer **tú**. No se calcula comparando con la sección del diario; es tu juicio como humano.

In [ ]:
RUTA_AUD = DATOS / "auditoria_llm.csv"
if not RUTA_AUD.exists():
    aud = df.sample(20, random_state=SEED)[["titular", "bajada", "etiqueta_llm", "confianza"]].copy()
    aud["revision_humana"] = ""
    aud["comentario"] = ""
    aud.to_csv(RUTA_AUD, index=False, sep=";", encoding="utf-8-sig")  # ';' para que Excel en español lo abra bien
    print(f"Creado {RUTA_AUD}. Complétalo y vuelve a ejecutar esta celda.")

aud = pd.read_csv(RUTA_AUD, sep=";", encoding="utf-8-sig").fillna("")
respuesta = aud["revision_humana"].astype(str).str.strip().str.lower().str.replace("í", "i")
revisadas = respuesta.isin(["si", "no"])
if revisadas.sum() < 20:
    ACUERDO_N = None
    print(f"Auditoría incompleta: {revisadas.sum()}/20 filas revisadas. Completa datos/auditoria_llm.csv.")
else:
    ACUERDO_N = int((respuesta == "si").sum())
    print(f"ACUERDO LLM-HUMANO: {ACUERDO_N}/20 = {ACUERDO_N / 20:.0%}")
pd.set_option("display.max_colwidth", 120)
aud

## 3. Fine-tuning de DistilBERT multilingüe

- **Texto de entrada:** solo el titular (es lo que la app recibirá en vivo).
- **Etiquetas de entrenamiento:** las del LLM (supervisión débil). Se excluyen las filas donde el LLM no devolvió una categoría válida.
- **Validación:** se usa la etiqueta de referencia del conjunto `val` para elegir la mejor época por F1 macro. El **test no se toca**.
- Configuración: 3 épocas, batch 16, evaluación por época, `metric_for_best_model="f1_macro"`.

En CPU tarda aprox. 5–15 minutos. Si va muy lento, ejecuta este notebook en Google Colab con GPU T4.

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
set_seed(SEED)

MODELO_BASE = "distilbert-base-multilingual-cased"
ETIQUETA_ENTRENAMIENTO = "etiqueta_llm"   # supervisión débil (cámbialo a "etiqueta" para comparar)
etiqueta2id = {c: i for i, c in enumerate(CATEGORIAS)}
id2etiqueta = {i: c for c, i in etiqueta2id.items()}

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

class DatasetTitulares(torch.utils.data.Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tokenizador(list(textos), truncation=True, max_length=64)
        self.labels = [etiqueta2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ok = train_df[train_df[ETIQUETA_ENTRENAMIENTO].isin(CATEGORIAS)]
print(f"Train: {len(train_ok)} titulares (se excluyeron {len(train_df) - len(train_ok)} sin etiqueta válida) · Val: {len(val_df)}")
ds_train = DatasetTitulares(train_ok["titular"], train_ok[ETIQUETA_ENTRENAMIENTO])
ds_val = DatasetTitulares(val_df["titular"], val_df["etiqueta"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(CATEGORIAS), id2label=id2etiqueta, label2id=etiqueta2id)

parametros = dict(output_dir=str(MODELO_DIR), num_train_epochs=3, per_device_train_batch_size=16,
                  per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
                  save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1_macro",
                  greater_is_better=True, save_total_limit=1, logging_strategy="epoch",
                  report_to="none", seed=SEED)
try:
    args = TrainingArguments(eval_strategy="epoch", **parametros)
except TypeError:  # versiones antiguas de transformers
    args = TrainingArguments(evaluation_strategy="epoch", **parametros)

trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizador), compute_metrics=compute_metrics)
t0 = time.perf_counter()
trainer.train()
print(f"Entrenamiento: {(time.perf_counter() - t0) / 60:.1f} min")

trainer.save_model(str(MODELO_DIR))
tokenizador.save_pretrained(str(MODELO_DIR))
print("Modelo guardado en modelo/ (los pesos .safetensors están en .gitignore)")

In [ ]:
# Curva de aprendizaje por época (solo train y validación)
hist = pd.DataFrame(trainer.state.log_history)
por_epoca = hist.groupby("epoch").agg({c: "last" for c in ["loss", "eval_loss", "eval_f1_macro"] if c in hist}).reset_index()
display(por_epoca.round(3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
por_epoca.plot(x="epoch", y=[c for c in ["loss", "eval_loss"] if c in por_epoca], marker="o", ax=ax[0], title="Pérdida por época")
por_epoca.plot(x="epoch", y="eval_f1_macro", marker="o", ax=ax[1], title="F1 macro en validación", legend=False)
plt.tight_layout(); plt.show()

## 4. Evaluación final en el conjunto de TEST

Es la **primera y única vez** que el modelo ve el test. La referencia es la sección del diario (`etiqueta`).

In [ ]:
clasificador = pipeline("text-classification", model=str(MODELO_DIR), tokenizer=str(MODELO_DIR), device=-1)

t0 = time.perf_counter()
pred_test = clasificador(test_df["titular"].tolist(), batch_size=16, truncation=True)
lat_ft = (time.perf_counter() - t0) / len(test_df) * 1000
test_df["pred_ft"] = [p["label"] for p in pred_test]
test_df["prob_ft"] = [round(p["score"], 3) for p in pred_test]

acc_ft = accuracy_score(test_df["etiqueta"], test_df["pred_ft"])
f1_ft = f1_score(test_df["etiqueta"], test_df["pred_ft"], average="macro")
print(f"FINE-TUNED en TEST ({len(test_df)} titulares) -> Accuracy: {acc_ft:.3f} | F1 macro: {f1_ft:.3f}")
print(classification_report(test_df["etiqueta"], test_df["pred_ft"], labels=CATEGORIAS, zero_division=0))

cm = confusion_matrix(test_df["etiqueta"], test_df["pred_ft"], labels=CATEGORIAS)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
plt.xlabel("Predicción del modelo"); plt.ylabel("Categoría real (sección del diario)")
plt.title(f"Matriz de confusión en TEST · F1 macro = {f1_ft:.2f}")
plt.tight_layout(); plt.show()
test_df[["titular", "etiqueta", "etiqueta_llm", "pred_ft", "prob_ft"]].to_csv(DATOS / "predicciones_test.csv", index=False)

In [ ]:
# Desempeño por categoría y errores, para la reflexión
reporte = pd.DataFrame(classification_report(test_df["etiqueta"], test_df["pred_ft"], labels=CATEGORIAS,
                                             output_dict=True, zero_division=0)).T.loc[CATEGORIAS]
reporte = reporte[["precision", "recall", "f1-score", "support"]].round(3).sort_values("f1-score", ascending=False)
display(reporte)
print(f"Mejor categoría: {reporte.index[0]} (F1 {reporte['f1-score'].iloc[0]:.2f}) · "
      f"Peor categoría: {reporte.index[-1]} (F1 {reporte['f1-score'].iloc[-1]:.2f})")
print("\nTitulares mal clasificados en test:")
test_df.loc[test_df["pred_ft"] != test_df["etiqueta"], ["titular", "etiqueta", "pred_ft", "prob_ft"]]

### Reflexión por categoría

> ✍️ **Edita este texto con TUS resultados** (mira la tabla y los errores de la celda anterior). Abajo hay hipótesis típicas que puedes confirmar o descartar:

- **Mejor desempeño — probablemente `deportes`:** su vocabulario es muy distintivo (nombres de clubes como Alianza Lima o Universitario, "gol", "Liga 1", "Selección") y casi no se comparte con otras secciones.
- **Peor desempeño — probablemente la frontera `politica` ↔ `economia`:** muchos titulares hablan de presupuesto, AFP, impuestos o medidas del Gobierno y del Congreso; el tema es económico pero el actor es político, y la sección del diario a veces decide por el actor.
- **`ciencia_tecnologia`:** en La República esta sección mezcla hallazgos científicos con curiosidades ("Científicos descubren...", "NASA..."), lo que suele dar titulares fáciles de reconocer; los errores aparecen cuando la noticia toca salud pública o medio ambiente con un ángulo político.
- **Efecto de la supervisión débil:** el modelo aprende de las etiquetas del LLM; cuando el LLM y la sección del diario discrepan, el modelo hereda ese criterio y "pierde" puntos contra la sección.

## 5. Comparativa y decisión

In [ ]:
# Las 3 estrategias evaluadas sobre los MISMOS 40 titulares del test y, cuando se puede, sobre el test completo
zs_muestra = zs_muestra.drop(columns=[c for c in ["etiqueta_llm", "pred_ft"] if c in zs_muestra.columns])
zs_muestra = zs_muestra.merge(test_df[["titular", "etiqueta_llm", "pred_ft"]], on="titular", how="left")
f1_40 = lambda col: f1_score(zs_muestra["etiqueta"], zs_muestra[col], average="macro", labels=CATEGORIAS)

f1_llm_test = f1_score(test_df["etiqueta"], test_df["etiqueta_llm"], average="macro", labels=CATEGORIAS)
acc_llm_test = accuracy_score(test_df["etiqueta"], test_df["etiqueta_llm"])

comparativa = pd.DataFrame([
    {"modelo": f"Zero-shot ({MODELO_ZS.split('/')[-1]})", "F1 macro (40 titulares test)": f1_40("pred_zero_shot"),
     "F1 macro (test completo)": np.nan, "Accuracy (test completo)": np.nan,
     "latencia ms/titular (CPU)": lat_zs, "costo": "Gratis, local; modelo ~440 MB"},
    {"modelo": f"LLM débil ({llm_df['modelo_llm'].iloc[0]})", "F1 macro (40 titulares test)": f1_40("etiqueta_llm"),
     "F1 macro (test completo)": f1_llm_test, "Accuracy (test completo)": acc_llm_test,
     "latencia ms/titular (CPU)": lat_llm, "costo": "API externa: cuota/pago por token, requiere internet"},
    {"modelo": "DistilBERT fine-tuned", "F1 macro (40 titulares test)": f1_40("pred_ft"),
     "F1 macro (test completo)": f1_ft, "Accuracy (test completo)": acc_ft,
     "latencia ms/titular (CPU)": lat_ft, "costo": "Gratis, local; ~540 MB, entrenamiento de minutos"},
]).round(3)
comparativa

### ¿Qué modelo usaría en un entorno real?

> ✍️ **Ajusta este texto a tus números** de la tabla anterior.

Para un clasificador en producción (por ejemplo, etiquetar automáticamente cada titular nuevo del diario) elegiría el **DistilBERT fine-tuned**:

- **Precisión:** supera claramente al zero-shot (el piso) y queda cerca del LLM, del que aprendió.
- **Costo:** corre en CPU, sin pagar por llamada ni depender de una API externa; el LLM cobra por token y tiene límites de uso.
- **Latencia:** responde en milisegundos por titular, mientras que el LLM tarda segundos por lote y depende de la red.

El **LLM** sigue siendo útil como **etiquetador** (supervisión débil) para crear o ampliar el dataset de entrenamiento, y el **zero-shot** sirve para arrancar cuando no hay ningún dato etiquetado.

In [ ]:
# Métricas que usa la app de Streamlit (archivo liviano, sí se sube a GitHub)
metricas = {
    "f1_macro_test": round(f1_ft, 4), "accuracy_test": round(acc_ft, 4),
    "f1_macro_zero_shot": round(f1_zs, 4), "accuracy_zero_shot": round(acc_zs, 4),
    "f1_macro_llm_test": round(f1_llm_test, 4),
    "acuerdo_llm_humano": ACUERDO_N, "auditoria_n": 20,
    "modelo_base": MODELO_BASE, "modelo_zero_shot": MODELO_ZS, "modelo_llm": str(llm_df["modelo_llm"].iloc[0]),
    "n_train": len(train_ok), "n_val": len(val_df), "n_test": len(test_df),
    "fecha": datetime.now().strftime("%Y-%m-%d"),
}
(MODELO_DIR / "metricas.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")
comparativa.to_csv(MODELO_DIR / "comparativa.csv", index=False)
acuerdo_txt = f"{ACUERDO_N}/20" if ACUERDO_N is not None else "PENDIENTE (completa la auditoría)"
print("Resultados para el Issue de entrega:")
print(f"Resultados Test: F1 Macro: {f1_ft:.2f} | Zero-shot: {f1_zs:.2f} | Acuerdo LLM-Humano: {acuerdo_txt}")

In [ ]:
# requirements.txt con las versiones EXACTAS que usaste (así es reproducible)
import importlib.metadata as md
paquetes = ["streamlit", "pandas", "numpy", "plotly", "kaleido", "requests", "beautifulsoup4", "lxml",
            "scikit-learn", "matplotlib", "seaborn", "torch", "transformers", "accelerate",
            "google-genai", "python-dotenv", "ipykernel"]
lineas = []
for p in paquetes:
    try:
        lineas.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError:
        print("No instalado (se omite):", p)
Path("requirements.txt").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(Path("requirements.txt").read_text())